In [ ]:
"""
This script performs automated hyperparameter tuning for Word2Vec model
using Optuna framework. It optimizes vector_size, window, and min_count
parameters to maximize Spearman correlation on WordSim353 dataset.
"""

import logging
import multiprocessing
import os
from time import time

import gensim
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from gensim.models import word2vec
from scipy.stats import spearmanr

# Configure logging to reduce verbosity during optimization
logging.basicConfig(
    format="%(asctime)s : %(levelname)s : %(message)s", level=logging.WARNING
)

# Global variable to store training history across trials
training_history = []

# Output directory for all results
OUTPUT_DIR = "optuna_results"


def setup_output_directory():
    """
    Create output directory if it does not exist.

    Returns:
        str: Path to the output directory
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output directory: {OUTPUT_DIR}")
    return OUTPUT_DIR


def load_wordsim353():
    """
    Load WordSim353 evaluation dataset.

    The WordSim353 dataset contains 353 word pairs with human-assigned
    similarity scores ranging from 0 to 10.

    Returns:
        list: List of tuples containing (word1, word2, human_score)
    """
    word_pairs = []
    with open("wordsim353/combined.csv") as f:
        for line in f.readlines()[1:]:  # Skip header
            parts = line.strip().split(",")
            word_pairs.append((parts[0], parts[1], float(parts[2])))
    return word_pairs


def evaluate_model(model, word_pairs):
    """
    Evaluate Word2Vec model using Spearman correlation coefficient.

    Computes similarity scores for word pairs present in the model's
    vocabulary and calculates Spearman correlation with human scores.

    Args:
        model: Trained gensim Word2Vec model
        word_pairs: List of (word1, word2, human_score) tuples

    Returns:
        float: Spearman correlation coefficient, or -1.0 if insufficient data
    """
    model_sims = []
    human_scores = []

    for w1, w2, score in word_pairs:
        # Only evaluate pairs where both words are in vocabulary
        if w1 in model.wv.key_to_index and w2 in model.wv.key_to_index:
            model_sims.append(model.wv.similarity(w1, w2))
            human_scores.append(score)

    # Require minimum number of samples for reliable correlation
    if len(model_sims) < 10:
        return -1.0

    spearman_corr, _ = spearmanr(human_scores, model_sims)
    return spearman_corr


def objective(trial):
    """
    Optuna objective function for hyperparameter optimization.

    Defines the search space for hyperparameters (vector_size, window, min_count),
    trains a Word2Vec model with default settings for other parameters,
    and returns the Spearman correlation as the optimization target.

    Args:
        trial: Optuna trial object for suggesting hyperparameters

    Returns:
        float: Spearman correlation coefficient (to be maximized)
    """
    # Define hyperparameter search space for only three parameters
    # Search around the initial values: vector_size=220, window=48, min_count=37
    vector_size = trial.suggest_int("vector_size", 50, 500, step=10)
    window = trial.suggest_int("window", 3, 50)
    min_count = trial.suggest_int("min_count", 2, 50)

    # Display trial information
    print(f"\n{'=' * 60}")
    print(
        f"Trial {trial.number}: vector_size={vector_size}, window={window}, "
        f"min_count={min_count}"
    )
    print("=" * 60)

    t_start = time()

    # Load dataset
    sents = word2vec.Text8Corpus("text8")

    # Train Word2Vec model with suggested hyperparameters
    # Use default values for all other parameters (sg=0 for CBOW, negative=5, etc.)
    model = gensim.models.Word2Vec(
        sents,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=multiprocessing.cpu_count(),
        epochs=5,  # Reduced epochs for faster tuning
    )

    train_time = time() - t_start

    # Evaluate model performance
    word_pairs = load_wordsim353()
    spearman_corr = evaluate_model(model, word_pairs)

    # Record trial results for later analysis
    training_history.append(
        {
            "trial": trial.number,
            "vector_size": vector_size,
            "window": window,
            "min_count": min_count,
            "spearman": spearman_corr,
            "train_time": train_time,
            "vocab_size": len(model.wv.key_to_index),
        }
    )

    print(
        f"Result: Spearman={spearman_corr:.4f}, Time={train_time:.1f}s, "
        f"Vocabulary={len(model.wv.key_to_index)}"
    )

    return spearman_corr


def plot_optimization_history(history_df):
    """
    Plot optimization progress over trials.

    Creates a line plot showing Spearman correlation for each trial
    and the cumulative best score.

    Args:
        history_df: DataFrame containing trial history
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    trials = range(len(history_df))
    ax.plot(
        trials,
        history_df["spearman"],
        "b-o",
        alpha=0.7,
        markersize=6,
        label="Each Trial",
    )
    ax.plot(
        trials, history_df["spearman"].cummax(), "r-", linewidth=2, label="Best So Far"
    )

    ax.set_xlabel("Trial Number", fontsize=12)
    ax.set_ylabel("Spearman Correlation", fontsize=12)
    ax.set_title("Optimization Progress", fontsize=14)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, "optimization_history.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


def plot_parameter_effects(history_df):
    """
    Plot the effect of each hyperparameter on model performance.

    Creates scatter plots showing relationship between each
    hyperparameter and Spearman correlation.

    Args:
        history_df: DataFrame containing trial history
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Vector size effect
    ax1 = axes[0, 0]
    scatter1 = ax1.scatter(
        history_df["vector_size"],
        history_df["spearman"],
        c=history_df["trial"],
        cmap="viridis",
        alpha=0.7,
        s=60,
    )
    ax1.set_xlabel("Vector Size", fontsize=11)
    ax1.set_ylabel("Spearman Correlation", fontsize=11)
    ax1.set_title("Effect of Vector Size", fontsize=12)
    plt.colorbar(scatter1, ax=ax1, label="Trial")
    ax1.grid(True, alpha=0.3)

    # Window size effect
    ax2 = axes[0, 1]
    scatter2 = ax2.scatter(
        history_df["window"],
        history_df["spearman"],
        c=history_df["trial"],
        cmap="viridis",
        alpha=0.7,
        s=60,
    )
    ax2.set_xlabel("Window Size", fontsize=11)
    ax2.set_ylabel("Spearman Correlation", fontsize=11)
    ax2.set_title("Effect of Window Size", fontsize=12)
    plt.colorbar(scatter2, ax=ax2, label="Trial")
    ax2.grid(True, alpha=0.3)

    # Min count effect
    ax3 = axes[1, 0]
    scatter3 = ax3.scatter(
        history_df["min_count"],
        history_df["spearman"],
        c=history_df["trial"],
        cmap="viridis",
        alpha=0.7,
        s=60,
    )
    ax3.set_xlabel("Min Count", fontsize=11)
    ax3.set_ylabel("Spearman Correlation", fontsize=11)
    ax3.set_title("Effect of Min Count", fontsize=12)
    plt.colorbar(scatter3, ax=ax3, label="Trial")
    ax3.grid(True, alpha=0.3)

    # Training time vs performance
    ax4 = axes[1, 1]
    scatter4 = ax4.scatter(
        history_df["train_time"],
        history_df["spearman"],
        c=history_df["vector_size"],
        cmap="plasma",
        alpha=0.7,
        s=60,
    )
    ax4.set_xlabel("Training Time (seconds)", fontsize=11)
    ax4.set_ylabel("Spearman Correlation", fontsize=11)
    ax4.set_title("Time vs Performance Trade-off", fontsize=12)
    plt.colorbar(scatter4, ax=ax4, label="Vector Size")
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, "parameter_effects.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


def plot_parameter_importance(study):
    """
    Plot hyperparameter importance analysis.

    Uses Optuna's built-in importance calculation to determine
    which hyperparameters have the most impact on performance.

    Args:
        study: Optuna study object
    """
    try:
        importance = optuna.importance.get_param_importances(study)

        fig, ax = plt.subplots(figsize=(10, 6))

        params = list(importance.keys())
        values = list(importance.values())

        bars = ax.barh(params, values, color="steelblue", edgecolor="navy")
        ax.set_xlabel("Importance Score", fontsize=12)
        ax.set_title("Hyperparameter Importance Analysis", fontsize=14)
        ax.grid(True, alpha=0.3, axis="x")

        # Add value labels on bars
        for bar, val in zip(bars, values):
            ax.text(
                val + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}",
                va="center",
                fontsize=10,
            )

        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, "parameter_importance.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {save_path}")

    except Exception as e:
        print(f"Warning: Could not compute parameter importance: {e}")


def plot_parameter_interactions(study):
    """
    Plot parameter interaction scatter plots.

    Creates scatter plots showing how combinations of parameters
    affect the optimization objective.

    Args:
        study: Optuna study object
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    param_pairs = [
        ("vector_size", "window"),
        ("vector_size", "min_count"),
        ("window", "min_count"),
    ]

    # Extract trial data
    trials_data = []
    for trial in study.trials:
        if trial.state == optuna.trial.TrialState.COMPLETE:
            trials_data.append(
                {
                    "vector_size": trial.params["vector_size"],
                    "window": trial.params["window"],
                    "min_count": trial.params["min_count"],
                    "value": trial.value,
                }
            )

    df = pd.DataFrame(trials_data)

    for ax, (param1, param2) in zip(axes, param_pairs):
        scatter = ax.scatter(
            df[param1],
            df[param2],
            c=df["value"],
            cmap="RdYlGn",
            s=100,
            alpha=0.7,
            edgecolors="black",
            linewidth=0.5,
        )
        ax.set_xlabel(param1, fontsize=11)
        ax.set_ylabel(param2, fontsize=11)
        ax.set_title(f"{param1} vs {param2}", fontsize=12)
        plt.colorbar(scatter, ax=ax, label="Spearman")

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, "parameter_interactions.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


def train_final_model(best_params, epochs=10):
    """
    Train the final model using optimized hyperparameters.

    Trains a Word2Vec model with more epochs using the best
    hyperparameters found during optimization. Only vector_size,
    window, and min_count are optimized; other parameters use defaults.

    Args:
        best_params: Dictionary containing best hyperparameters
        epochs: Number of training epochs for final model

    Returns:
        tuple: (trained_model, spearman_correlation, training_time)
    """
    print("\n" + "=" * 60)
    print("Training Final Model with Optimized Parameters")
    print("=" * 60)

    # Display optimized parameters
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    print(f"  epochs: {epochs}")
    print("  (other parameters use default values)")

    t_start = time()

    # Load dataset
    sents = word2vec.Text8Corpus("text8")

    # Train model with best parameters (only vector_size, window, min_count)
    # All other parameters use gensim defaults
    model = gensim.models.Word2Vec(
        sents,
        vector_size=best_params["vector_size"],
        window=best_params["window"],
        min_count=best_params["min_count"],
        workers=multiprocessing.cpu_count(),
        epochs=epochs,
    )

    train_time = time() - t_start

    # Save model files
    model_path = os.path.join(OUTPUT_DIR, "word2vec_best_model")
    vectors_path = os.path.join(OUTPUT_DIR, "word2vec_vectors.txt")
    vocab_path = os.path.join(OUTPUT_DIR, "vocabulary.txt")

    model.save(model_path)
    model.wv.save_word2vec_format(vectors_path, vocab_path, binary=False)

    # Evaluate final model
    word_pairs = load_wordsim353()
    spearman_corr = evaluate_model(model, word_pairs)

    # Save evaluation scores
    sims = []
    ground_truth = []
    for w1, w2, score in word_pairs:
        if w1 in model.wv.key_to_index and w2 in model.wv.key_to_index:
            sims.append(model.wv.similarity(w1, w2))
            ground_truth.append(score)

    np.save(os.path.join(OUTPUT_DIR, "model_scores.npy"), np.array(sims))
    np.save(os.path.join(OUTPUT_DIR, "ground_truth.npy"), np.array(ground_truth))

    print(f"\nModel saved to: {model_path}")
    print(f"Vectors saved to: {vectors_path}")
    print(f"Training time: {train_time:.1f}s")
    print(f"Final Spearman correlation: {spearman_corr:.4f}")

    return model, spearman_corr, train_time


def generate_report(study, history_df, final_spearman, final_time):
    """
    Generate comprehensive optimization report.

    Creates a text report summarizing the optimization results,
    best parameters, and performance statistics.

    Args:
        study: Optuna study object
        history_df: DataFrame containing trial history
        final_spearman: Final model's Spearman correlation
        final_time: Final model's training time
    """
    lines = []
    lines.append("=" * 60)
    lines.append("    Word2Vec Hyperparameter Optimization Report")
    lines.append("=" * 60)

    lines.append("\n[Optimized Hyperparameters]")
    lines.append("  (Only vector_size, window, min_count are optimized)")
    for param, value in study.best_params.items():
        lines.append(f"  {param}: {value}")

    lines.append("\n[Performance Metrics]")
    lines.append(f"  Best Spearman (tuning phase): {study.best_value:.4f}")
    lines.append(f"  Final Spearman (full training): {final_spearman:.4f}")
    lines.append(f"  Final training time: {final_time:.1f}s")

    lines.append("\n[Search Statistics]")
    lines.append(f"  Total trials: {len(study.trials)}")
    lines.append(f"  Mean Spearman: {history_df['spearman'].mean():.4f}")
    lines.append(f"  Std Spearman: {history_df['spearman'].std():.4f}")
    lines.append(f"  Min Spearman: {history_df['spearman'].min():.4f}")
    lines.append(f"  Max Spearman: {history_df['spearman'].max():.4f}")
    lines.append(f"  Total search time: {history_df['train_time'].sum():.1f}s")

    lines.append("\n[Top 5 Trials]")
    top5 = history_df.nlargest(5, "spearman")
    lines.append(
        top5[["trial", "vector_size", "window", "min_count", "spearman"]].to_string(
            index=False
        )
    )

    lines.append("\n[Recommended Parameter Ranges]")
    best_trials = history_df.nlargest(5, "spearman")
    for param in ["vector_size", "window", "min_count"]:
        min_val = best_trials[param].min()
        max_val = best_trials[param].max()
        lines.append(f"  {param}: [{min_val}, {max_val}]")

    # Print report
    report_text = "\n".join(lines)
    print("\n" + report_text)

    # Save report to file
    report_path = os.path.join(OUTPUT_DIR, "optimization_report.txt")
    with open(report_path, "w") as f:
        f.write(report_text)
    print(f"\nReport saved to: {report_path}")


def test_model_quality(model):
    """
    Test model quality with word similarity and analogy tasks.

    Demonstrates the trained model's ability to capture semantic
    relationships through similar word retrieval and word analogies.

    Args:
        model: Trained Word2Vec model
    """
    print("\n" + "=" * 60)
    print("Model Quality Tests")
    print("=" * 60)

    # Word similarity test
    print("\n[Most Similar Words]")
    test_words = ["king", "queen", "computer", "science", "good", "bad"]

    for word in test_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=5)
            similar_str = ", ".join([f"{w}({s:.2f})" for w, s in similar])
            print(f"  {word}: {similar_str}")
        else:
            print(f"  {word}: [not in vocabulary]")

    # Word analogy test
    print("\n[Word Analogies]")
    analogies = [
        (["king", "woman"], ["man"], "king - man + woman"),
        (["paris", "germany"], ["france"], "paris - france + germany"),
        (["walking", "walked"], ["swimming"], "walking - walked + swimming"),
    ]

    for positive, negative, description in analogies:
        all_words = positive + negative
        if all(w in model.wv for w in all_words):
            try:
                result = model.wv.most_similar(
                    positive=positive, negative=negative, topn=3
                )
                result_str = ", ".join([f"{w}({s:.2f})" for w, s in result])
                print(f"  {description} = {result_str}")
            except Exception as e:
                print(f"  {description} = [error: {e}]")
        else:
            missing = [w for w in all_words if w not in model.wv]
            print(f"  {description} = [missing words: {missing}]")


def main():
    """
    Main function to run the complete optimization pipeline.

    Executes the following steps:
    1. Setup output directory
    2. Run Optuna hyperparameter optimization (only vector_size, window, min_count)
    3. Generate visualization plots
    4. Train final model with best parameters
    5. Generate optimization report
    6. Test model quality
    """
    print("=" * 60)
    print("  Word2Vec Hyperparameter Optimization with Optuna")
    print("  (Optimizing: vector_size, window, min_count)")
    print("=" * 60)

    # Setup
    setup_output_directory()

    # Create Optuna study
    study = optuna.create_study(
        direction="maximize",  # Maximize Spearman correlation
        study_name="word2vec_optimization",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    # Run optimization
    n_trials = 50  # Adjust based on available time
    print(f"\nStarting optimization with {n_trials} trials...")
    print("Optimizing only: vector_size, window, min_count")
    print("Other parameters use default values.")
    print("This may take a while depending on your hardware.\n")

    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    # Save optimization history
    history_df = pd.DataFrame(training_history)
    history_path = os.path.join(OUTPUT_DIR, "optimization_history.csv")
    history_df.to_csv(history_path, index=False)
    print(f"\nOptimization history saved to: {history_path}")

    # Generate all plots
    print("\nGenerating visualization plots...")
    plot_optimization_history(history_df)
    plot_parameter_effects(history_df)
    plot_parameter_importance(study)
    plot_parameter_interactions(study)

    # Train final model
    final_model, final_spearman, final_time = train_final_model(
        study.best_params, epochs=10
    )

    # Generate report
    generate_report(study, history_df, final_spearman, final_time)

    # Test model quality
    test_model_quality(final_model)

    # Summary
    print("\n" + "=" * 60)
    print("Optimization Complete!")
    print("=" * 60)
    print(f"All results saved to: {OUTPUT_DIR}/")
    print("\nGenerated files:")
    for filename in os.listdir(OUTPUT_DIR):
        filepath = os.path.join(OUTPUT_DIR, filename)
        size = os.path.getsize(filepath)
        if size > 1024 * 1024:
            size_str = f"{size / (1024 * 1024):.1f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"  - {filename} ({size_str})")


if __name__ == "__main__":
    main()